[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/LegalIntermediaSL/InteligenciaArtificial/blob/main/tutoriales/notebooks/ia-empresarial/01-estrategia-ia.ipynb)

# Estrategia de IA para Empresas

Implementamos herramientas para evaluar casos de uso, calcular ROI y priorizar iniciativas de IA.

In [ ]:
import anthropic
import json

client = anthropic.Anthropic()
print("Herramientas de estrategia de IA listas")

## 1. Evaluador de casos de uso

In [ ]:
def evaluar_caso_uso(descripcion: str, contexto_empresa: str) -> dict:
    """Evalúa un caso de uso de IA y devuelve análisis estructurado."""

    resp = client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=1000,
        messages=[{
            "role": "user",
            "content": f"""Evalúa este caso de uso de IA para una empresa:

CASO DE USO: {descripcion}
CONTEXTO EMPRESA: {contexto_empresa}

Responde en JSON con esta estructura:
{{
  "viabilidad": "alta/media/baja",
  "impacto_negocio": "alto/medio/bajo",
  "complejidad_tecnica": "alta/media/baja",
  "tiempo_implementacion_semanas": 8,
  "riesgos_principales": ["riesgo 1", "riesgo 2"],
  "kpis_sugeridos": ["kpi 1", "kpi 2"],
  "recomendacion": "texto breve con la recomendación"
}}"""
        }]
    )

    texto = resp.content[0].text
    if "```" in texto:
        texto = texto.split("```")[1].lstrip("json\n")
    try:
        return json.loads(texto)
    except json.JSONDecodeError:
        return {"error": texto}


# Evaluamos 3 casos de uso típicos
contexto = "SaaS B2B, 50 empleados, 800 clientes, equipo de datos de 2 personas, presupuesto tech limitado"

casos = [
    "Automatizar la clasificación y respuesta de emails de soporte al cliente",
    "Generar borradores de propuestas comerciales personalizadas",
    "Predecir qué clientes van a cancelar su suscripción en los próximos 30 días"
]

resultados = []
for caso in casos:
    print(f"\nEvaluando: {caso}")
    analisis = evaluar_caso_uso(caso, contexto)
    analisis["descripcion"] = caso
    resultados.append(analisis)
    print(f"  Viabilidad: {analisis.get('viabilidad', 'N/A')} | Impacto: {analisis.get('impacto_negocio', 'N/A')}")
    print(f"  Tiempo: {analisis.get('tiempo_implementacion_semanas', '?')} semanas")
    print(f"  Recomendación: {analisis.get('recomendacion', 'N/A')[:100]}")

## 2. Matriz de priorización

In [ ]:
def priorizar_casos_uso(casos_analizados: list) -> list:
    """Ordena casos de uso por puntuación impacto/esfuerzo."""
    mapa_puntos = {"alto": 3, "alta": 3, "medio": 2, "media": 2, "bajo": 1, "baja": 1}

    for caso in casos_analizados:
        impacto = mapa_puntos.get(caso.get("impacto_negocio", "bajo"), 1)
        viabilidad = mapa_puntos.get(caso.get("viabilidad", "baja"), 1)
        complejidad_inv = 4 - mapa_puntos.get(caso.get("complejidad_tecnica", "alta"), 3)
        caso["puntuacion"] = impacto * viabilidad * complejidad_inv

    return sorted(casos_analizados, key=lambda x: x["puntuacion"], reverse=True)


priorizados = priorizar_casos_uso(resultados)

print("RANKING DE CASOS DE USO")
print("=" * 60)
for i, caso in enumerate(priorizados, 1):
    print(f"\n#{i} (puntuación: {caso.get('puntuacion', 0)})")
    print(f"  {caso['descripcion']}")
    print(f"  Viabilidad: {caso.get('viabilidad')} | Impacto: {caso.get('impacto_negocio')} | Complejidad: {caso.get('complejidad_tecnica')}")
    kpis = caso.get('kpis_sugeridos', [])
    if kpis:
        print(f"  KPIs: {', '.join(kpis[:2])}")

## 3. Calculadora de ROI

In [ ]:
def calcular_roi_ia(
    horas_ahorradas_semana: float,
    coste_hora_empleado: float,
    coste_api_mensual: float,
    coste_implementacion: float,
    semanas_por_mes: float = 4.33
) -> dict:
    """Calcula el ROI de una implementación de IA."""
    ahorro_mensual = horas_ahorradas_semana * semanas_por_mes * coste_hora_empleado
    beneficio_neto_mensual = ahorro_mensual - coste_api_mensual

    if beneficio_neto_mensual > 0:
        meses_recuperacion = coste_implementacion / beneficio_neto_mensual
        roi_anual = ((beneficio_neto_mensual * 12) - coste_implementacion) / coste_implementacion * 100
    else:
        meses_recuperacion = float("inf")
        roi_anual = -100

    return {
        "ahorro_bruto_mensual_eur": round(ahorro_mensual, 2),
        "coste_api_mensual_eur": coste_api_mensual,
        "beneficio_neto_mensual_eur": round(beneficio_neto_mensual, 2),
        "coste_implementacion_eur": coste_implementacion,
        "meses_para_recuperar": round(meses_recuperacion, 1) if meses_recuperacion != float("inf") else "nunca",
        "roi_anual_pct": round(roi_anual, 1),
        "viable": beneficio_neto_mensual > 0
    }


# Escenarios de ROI
escenarios = [
    {
        "nombre": "Automatización de soporte (equipo de 3 agentes)",
        "horas_ahorradas_semana": 20,
        "coste_hora_empleado": 35,
        "coste_api_mensual": 400,
        "coste_implementacion": 8000
    },
    {
        "nombre": "Generación de propuestas comerciales",
        "horas_ahorradas_semana": 8,
        "coste_hora_empleado": 50,
        "coste_api_mensual": 150,
        "coste_implementacion": 3000
    },
    {
        "nombre": "Clasificación de leads automática",
        "horas_ahorradas_semana": 5,
        "coste_hora_empleado": 45,
        "coste_api_mensual": 200,
        "coste_implementacion": 5000
    }
]

print("ANÁLISIS DE ROI POR ESCENARIO")
print("=" * 65)

for esc in escenarios:
    roi = calcular_roi_ia(
        esc["horas_ahorradas_semana"],
        esc["coste_hora_empleado"],
        esc["coste_api_mensual"],
        esc["coste_implementacion"]
    )
    print(f"\n{esc['nombre']}")
    print(f"  Inversión: {roi['coste_implementacion_eur']}€")
    print(f"  Ahorro mensual bruto: {roi['ahorro_bruto_mensual_eur']}€")
    print(f"  Coste API mensual: {roi['coste_api_mensual_eur']}€")
    print(f"  Beneficio neto mensual: {roi['beneficio_neto_mensual_eur']}€")
    print(f"  Recuperación: {roi['meses_para_recuperar']} meses")
    print(f"  ROI anual: {roi['roi_anual_pct']}%")

## 4. Plan de comunicación para la adopción

In [ ]:
def generar_plan_comunicacion(departamento: str, caso_uso: str, impacto_empleo: str = "reducción de tareas repetitivas") -> str:
    """Genera un plan de comunicación para la adopción de IA."""
    resp = client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=700,
        messages=[{
            "role": "user",
            "content": f"""Crea un plan de comunicación para introducir IA en {departamento}.
El caso de uso es: {caso_uso}
Impacto en el trabajo del equipo: {impacto_empleo}

El plan debe incluir:
1. Mensaje clave para el equipo (qué cambia, qué NO cambia)
2. Cómo presentarlo para reducir el miedo al reemplazo
3. Plan de formación básico (3 pasos)
4. Cómo medir la adopción en las primeras 4 semanas

Sé directo y práctico. Máximo 400 palabras."""
        }]
    )
    return resp.content[0].text


plan = generar_plan_comunicacion(
    "Customer Success",
    "asistente IA que sugiere respuestas a tickets y genera borradores automáticos",
    "reduce el tiempo por ticket un 40%, pero el CSM siempre revisa y envía"
)
print(plan)